# Executive Summary – Local Test Notebook

**Standalone** – does not import or reference `executive_summary_tab.py`.  
Contains identical logic (connection helpers, SQL, prompt, LLM call) so both
files can be maintained in sync independently.

**Environment auto-detection (same as the .py file)**
- If Airflow is importable → uses Airflow connections / variables (production).
- Otherwise → reads credentials from `os.environ` (local mode).

**Steps**
1. Cell 1 – install local deps if needed  
2. Cell 2 – set credentials (prompts via getpass)  
3. Cell 3 – all shared helpers (connection, SQL, prompt, LLM)  
4. Cell 4 – choose a month  
5. Cell 5 – query GP and inspect data  
6. Cell 6 – generate summary  
7. Cell 7 – save to .txt (optional)

In [1]:
# ── Cell 1: Install local dependencies (run once if needed) ──────────────
# !pip install psycopg2-binary openai pandas

In [2]:
# ── Cell 2: Credentials ───────────────────────────────────────────────────
#
# Set env vars BEFORE Cell 3 runs so the connection helpers pick them up.
# When Airflow IS available (production) these env vars are ignored.

import os
import getpass

os.environ['GP_HOST']     = 'greenplum-rdsp.zur.swissbank.com'
os.environ['GP_PORT']     = '5432'
os.environ['GP_DB']       = 'gprdsp'
os.environ['GP_USER']     = 'ds_rdsp_dev'
os.environ['GP_SCHEMA']   = 'core_ikg'
os.environ['GP_PASSWORD'] = getpass.getpass('Greenplum password: ')

os.environ['OPENAI_API_KEY']  = getpass.getpass('OpenAI / Azure API key: ')
os.environ['OPENAI_BASE_URL'] = 'https://cirruspl-staat-ste-dev-ai.openai.azure.com/openai/v1/'

print('Credentials stored in environment variables.')

Greenplum password:  ········
OpenAI / Azure API key:  ········


Credentials stored in environment variables.


In [3]:
# ── Cell 3: All shared helpers (identical logic to executive_summary_tab.py) ─

from __future__ import annotations

import os
from datetime import datetime
from typing import Optional

import pandas as pd
from openai import OpenAI


# ── Environment detection ──────────────────────────────────────────────────

try:
    from airflow.models import Connection, Variable                      # type: ignore
    from airflow.providers.postgres.hooks.postgres import PostgresHook  # type: ignore
    _AIRFLOW_AVAILABLE = True
except ImportError:
    _AIRFLOW_AVAILABLE = False

print(f'Airflow available: {_AIRFLOW_AVAILABLE}')
print('Mode:', 'PRODUCTION (Airflow connections)' if _AIRFLOW_AVAILABLE else 'LOCAL (env vars)')


# ── Model constants ────────────────────────────────────────────────────────

MODEL_NAME  = 'gpt-4.1'
MAX_TOKENS  = 12000
TEMPERATURE = 0.1

_client: Optional[OpenAI] = None


# ── Connection helpers ─────────────────────────────────────────────────────

def _get_openai_client() -> OpenAI:
    """Return shared OpenAI client.
    Production : Airflow Connection 'STAAT-DS-OPENAI-LLM'
    Local      : env vars OPENAI_API_KEY / OPENAI_BASE_URL
    """
    global _client
    if _client is not None:
        return _client
    if _AIRFLOW_AVAILABLE:
        conn     = Connection.get_connection_from_secrets('STAAT-DS-OPENAI-LLM')
        api_key  = conn.password
        base_url = conn.host
    else:
        api_key  = os.environ['OPENAI_API_KEY']
        base_url = os.environ.get(
            'OPENAI_BASE_URL',
            'https://cirruspl-staat-ste-dev-ai.openai.azure.com/openai/v1/',
        )
    _client = OpenAI(api_key=api_key, base_url=base_url)
    return _client


def _get_gp_conn():
    """Return a live GP connection.
    Production : PostgresHook via Airflow Variable 'GP_Dash_connect'
    Local      : psycopg2 using GP_* env vars
    """
    if _AIRFLOW_AVAILABLE:
        connect = Variable.get('GP_Dash_connect')
        return PostgresHook(postgres_conn_id=connect).get_conn()
    import psycopg2
    return psycopg2.connect(
        host    =os.environ['GP_HOST'],
        port    =int(os.environ.get('GP_PORT', 5432)),
        dbname  =os.environ['GP_DB'],
        user    =os.environ['GP_USER'],
        password=os.environ['GP_PASSWORD'],
    )


def _get_schema() -> str:
    """Return IKG schema name.
    Production : Airflow Variable 'IKG_DASHBOARD_SCHEMA'
    Local      : env var GP_SCHEMA (default 'core_ikg')
    """
    if _AIRFLOW_AVAILABLE:
        return Variable.get('IKG_DASHBOARD_SCHEMA')
    return os.environ.get('GP_SCHEMA', 'core_ikg')


def _get_excluded_insight_types(schema: str, conn) -> set:
    """Return the set of insight_type values currently excluded from the summary.

    Reads from {schema}.odm_exclusion_insight_type where is_curr = 1.
    Returns an empty set on any error so callers degrade gracefully.
    """
    sql = f"""
        SELECT insight_type
        FROM {schema}.odm_exclusion_insight_type
        WHERE is_curr = 1
    """
    try:
        df = pd.read_sql_query(sql, conn)
        return set(df['insight_type'].dropna().str.strip())
    except Exception:
        return set()


# ── SQL ────────────────────────────────────────────────────────────────────

def _build_month_query(month_value: str, schema: str) -> str:
    """SQL for latest batch per iteration whose prod_release_date is in month_value."""
    staat = f'{schema}.staat_insight_release'
    odm   = f'{schema}.odm_release_details'
    return f"""
    WITH target_iterations AS (
        SELECT iteration_end_date, MAX(batch) AS max_batch
        FROM {staat}
        WHERE TO_CHAR(prod_release_date::date, 'YYYY-MM') = '{month_value}'
        GROUP BY iteration_end_date
    ),
    staat_latest AS (
        SELECT s.*
        FROM {staat} s
        INNER JOIN target_iterations ti
            ON s.iteration_end_date = ti.iteration_end_date
            AND s.batch = ti.max_batch
    ),
    odm_latest AS (
        SELECT o.*
        FROM {odm} o
        INNER JOIN target_iterations ti
            ON o.iteration_end_date = ti.iteration_end_date
            AND o.batch = ti.max_batch
    )
    SELECT
        s.id_x, s.title, s.labels, s.issue_summary, s.state, s.weight,
        s.prod_release_date, s.iteration_end_date, s.iteration_start_date, s.batch,
        o.rule_name, o.target_type, o.change_type
    FROM staat_latest s
    LEFT JOIN odm_latest o
        ON s.id_x = o.issue_id
        AND s.iteration_end_date = o.iteration_end_date
    ORDER BY s.iteration_end_date, s.id_x
    """


def get_exec_summary_data(month_value: str) -> tuple:
    """Query GP; return (current_month_df, next_month_df)."""
    schema = _get_schema()
    conn   = _get_gp_conn()

    excluded = _get_excluded_insight_types(schema, conn)

    current_df = pd.read_sql_query(_build_month_query(month_value, schema), conn)
    current_df['change_type'] = current_df['change_type'].replace({'added': 'new'})
    if excluded:
        current_df = current_df[~current_df['rule_name'].isin(excluded)]

    try:
        next_period = str(pd.Period(month_value, 'M') + 1)
        next_df     = pd.read_sql_query(_build_month_query(next_period, schema), conn)
        next_df['change_type'] = next_df['change_type'].replace({'added': 'new'})
        if excluded:
            next_df = next_df[~next_df['rule_name'].isin(excluded)]
        next_month_df = next_df if not next_df.empty else None
    except Exception:
        next_month_df = None
    conn.close()
    return current_df, next_month_df


def get_month_options_from_db() -> list:
    """Return sorted dropdown options from GP."""
    schema = _get_schema()
    conn   = _get_gp_conn()
    sql    = f"""
        SELECT DISTINCT TO_CHAR(prod_release_date::date, 'YYYY-MM') AS month_val
        FROM {schema}.staat_insight_release
        WHERE prod_release_date IS NOT NULL
        ORDER BY month_val DESC
    """
    df = pd.read_sql_query(sql, conn)
    conn.close()
    options = []
    for val in df['month_val']:
        try:
            dt = pd.Period(val, 'M').to_timestamp()
            options.append({'label': dt.strftime('%B %Y'), 'value': val})
        except Exception:
            pass
    return options


# ── Label helpers ──────────────────────────────────────────────────────────

def _has_label(labels_value, target: str) -> bool:
    if not labels_value or not isinstance(labels_value, str):
        return False
    return target.lower() in [l.strip().lower() for l in labels_value.split(',')]


def _filter_by_label(df: pd.DataFrame, label: str) -> pd.DataFrame:
    if df is None or df.empty or 'labels' not in df.columns:
        return pd.DataFrame()
    return df.loc[df['labels'].apply(lambda v: _has_label(v, label))]


# ── Prompt ─────────────────────────────────────────────────────────────────

def _build_prompt(issues_df: pd.DataFrame, next_month_df=None) -> str:
    # Section 1: all stories
    issues_lines = []
    seen_ids: set = set()
    for _, row in issues_df.iterrows():
        id_x = row.get('id_x', '')
        if id_x and id_x not in seen_ids:
            seen_ids.add(id_x)
            issues_lines.append(
                f"Story #{len(issues_lines)+1}:\n"
                f"Title:   {row.get('title','N/A')}\n"
                f"Summary: {row.get('issue_summary','N/A')}\n"
                f"State:   {row.get('state','N/A')}\n"
                f"Labels:  {row.get('labels','N/A')}\n"
                f"Weight:  {row.get('weight','N/A')}"
            )
    issues_text = '\n\n'.join(issues_lines[:20]) if issues_lines else 'No issues data available.'

    # Section 2: Top Feature
    top_df    = _filter_by_label(issues_df, 'Top Feature')
    top_lines = []
    seen_top: set = set()
    for _, row in top_df.iterrows():
        id_x = row.get('id_x', '')
        if id_x and id_x not in seen_top:
            seen_top.add(id_x)
            top_lines.append(
                f"Top Feature #{len(top_lines)+1}:\n"
                f"Title:   {row.get('title','N/A')}\n"
                f"Summary: {row.get('issue_summary','N/A')}\n"
                f"Labels:  {row.get('labels','N/A')}\n"
                f"Rule:    {row.get('rule_name','N/A')}\n"
                f"Change:  {row.get('change_type','N/A')}"
            )
    top_feature_text = ('\n\n'.join(top_lines[:5]) if top_lines
                        else "No issue labelled 'Top Feature' found for this period.")

    # Section 3: New Insights
    ni_df     = _filter_by_label(issues_df, 'New Insight')
    ni_lines  = []
    seen_rules: set = set()
    for _, row in ni_df.iterrows():
        rule = row.get('rule_name', '')
        if rule and rule not in seen_rules:
            seen_rules.add(rule)
            ni_lines.append(
                f"New Insight #{len(ni_lines)+1}:\n"
                f"Rule Name:   {rule}\n"
                f"Target Type: {row.get('target_type','N/A')}\n"
                f"Change Type: {row.get('change_type','N/A')}\n"
                f"Title:       {row.get('title','N/A')}\n"
                f"Summary:     {row.get('issue_summary','N/A')}"
            )
    insights_text = ('\n\n'.join(ni_lines[:15]) if ni_lines
                     else "No issues labelled 'New Insight' found for this period.")

    # Section 6: Looking Ahead – New Insight titles from next month only
    next_lines = []
    if next_month_df is not None and not next_month_df.empty:
        next_ni_df = _filter_by_label(next_month_df, 'New Insight')
        next_seen: set = set()
        for _, row in next_ni_df.iterrows():
            rule  = row.get('rule_name', '')
            title = row.get('title', '')
            key   = rule or title
            if key and key not in next_seen:
                next_seen.add(key)
                next_lines.append(f"- {row.get('title', rule)}")
    next_month_text = ('\n'.join(next_lines[:10]) if next_lines
                       else "No next-month data found – provide directional focus areas based on current trends.")

    prompt = f"""
    You are an expert product manager and technical writer. Based on the following GitLab issues and
    new insights data, generate a comprehensive executive summary report.

    === GITLAB ISSUES DATA (all stories for this month) ===
    {issues_text}

    === TOP FEATURE DATA (issues labelled \"Top Feature\") ===
    {top_feature_text}

    === NEW INSIGHTS DATA (issues labelled \"New Insight\") ===
    {insights_text}

    === REQUIRED OUTPUT FORMAT ===
    Please analyse the data and provide a structured report. Use \"<Question>: <answer>\" style:

    1. Executive Summary:
    - What materially changed this month: <answer>
    - Why it matters to the business: <answer>

    2. Top Feature (or Insight) of the Month:
    - Insight name: <answer>
    - Brief description / example of narrative: <answer>
    - Main benefit / value: <answer>

    3. New Insights (ranked by importance/impact):
    For each new insight:
    - Insight name: <answer>
    - Brief description / example of narrative: <answer>
    - Main benefit / value: <answer>

    4. Process Improvements & Optimization:
    - Name / brief description: <answer>
    - How did we do it? (If AI was used, altered process, etc.): <answer>
    - Benefits: <answer>

    5. Platform Maintenance & Stability:
    - Maintenance, fixes, or technical improvements: <answer>
    - Why this matters (risk reduction, performance, cost control): <answer>

    6. New Insights \u2013 Coming Soon:
    Use the following next-month data (if available):
    {next_month_text}
    - Focus areas (2-4 items max): list Insight Title only, limited to \"New Insight\" labelled items

    === INSTRUCTIONS ===
    - Be concise and business-focused
    - Rank insights by business impact and value
    - Use clear, non-technical language where possible
    - Focus on outcomes and benefits, not just features
    - If certain sections have no relevant data, state \"No significant changes in this area\"
    - Ensure all answers are data-driven based on the provided information
    - Section 1 Executive Summary: 2 sentences per question
    - Section 2 Top Feature: pick the single most impactful item from the \"Top Feature\" labelled issues
    - Section 3 New Insights: list each individual insight on a separate line, drawn from \"New Insight\" labelled issues
    - Section 6 New Insights \u2013 Coming Soon: simple list of Insight Titles only, from the \"New Insight\" labelled items in the next month's current sprint. If no next-month data is available, provide directional focus areas based on current trends
    - Do NOT include instruction notes in the final output
    """
    return prompt.strip()


# ── LLM call ───────────────────────────────────────────────────────────────

def generate_summary(issues_df: pd.DataFrame, next_month_df=None) -> str:
    """Call the LLM and return the executive summary text."""
    client   = _get_openai_client()
    prompt   = _build_prompt(issues_df, next_month_df=next_month_df)
    response = client.chat.completions.create(
        model       =MODEL_NAME,
        messages    =[{'role': 'user', 'content': prompt}],
        max_tokens  =MAX_TOKENS,
        temperature =TEMPERATURE,
    )
    return response.choices[0].message.content.strip()


def build_save_content(summary_text: str, month_label: str) -> str:
    """Format summary for saving to a .txt file."""
    header = (
        f'Executive Summary Report \u2013 {month_label}\n'
        f'Generated on: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
        + '=' * 80 + '\n\n'
    )
    return header + summary_text


print('All helpers loaded.')


Airflow available: False
Mode: LOCAL (env vars)
All helpers loaded.


In [4]:
# ── Cell 4: Choose a month ────────────────────────────────────────────────

month_options = get_month_options_from_db()

print('Available months (latest first):')
for i, opt in enumerate(month_options, start=1):
    print(f"  {i:>2}. {opt['label']}  ({opt['value']})")

print()
raw = input("Enter the month to summarise (e.g. 'May 2026' or '2026-05'): ").strip()

try:
    SELECTED_MONTH = str(pd.Period(raw, 'M'))
except Exception:
    cleaned = raw.replace('-', ' ').replace('/', ' ')
    SELECTED_MONTH = datetime.strptime(cleaned, '%B %Y').strftime('%Y-%m')

SELECTED_LABEL = pd.Period(SELECTED_MONTH, 'M').to_timestamp().strftime('%B %Y')
print(f'\nSelected: {SELECTED_LABEL}  [{SELECTED_MONTH}]')

/tmp/ipykernel_117/3936540721.py:179: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


Available months (latest first):
   1. June 2026  (2026-06)
   2. May 2026  (2026-05)
   3. April 2026  (2026-04)
   4. March 2026  (2026-03)
   5. February 2026  (2026-02)
   6. January 2026  (2026-01)



Enter the month to summarise (e.g. 'May 2026' or '2026-05'):  2026-05



Selected: May 2026  [2026-05]


In [5]:
# ── Cell 5: Query GP and inspect data ─────────────────────────────────────

df_current, df_next_month = get_exec_summary_data(SELECTED_MONTH)

NEXT_LABEL = (pd.Period(SELECTED_MONTH, 'M') + 1).to_timestamp().strftime('%B %Y')
print(f'Current month ({SELECTED_LABEL}) rows : {len(df_current)}')
print(f'Next month    ({NEXT_LABEL}) rows : {len(df_next_month) if df_next_month is not None else 0}')
print()

display(df_current.head(10))

print('\nLabel breakdown:')
display(
    df_current['labels'].dropna().str.split(',').explode()
    .str.strip().value_counts()
)

print("\nRows labelled 'Top Feature':")
display(
    df_current[
        df_current['labels'].fillna('').apply(lambda v: _has_label(v, 'Top Feature'))
    ][['id_x','title','labels','rule_name','change_type']]
)

print("\nRows labelled 'New Insight':")
display(
    df_current[
        df_current['labels'].fillna('').apply(lambda v: _has_label(v, 'New Insight'))
    ][['id_x','title','labels','rule_name','change_type']]
)

/tmp/ipykernel_117/3936540721.py:99: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)
/tmp/ipykernel_117/3936540721.py:151: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  current_df = pd.read_sql_query(_build_month_query(month_value, schema), conn)
/tmp/ipykernel_117/3936540721.py:158: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  next_df     = pd.read_sql_query(_build_month_query(next_period, schema), conn)


Current month (May 2026) rows : 53
Next month    (June 2026) rows : 70



,id_x,title,labels,issue_summary,state,weight,prod_release_date,iteration_end_date,iteration_start_date,batch,rule_name,target_type,change_type
0,2006,Prospect Verify API solution,"Amol, IKG, Priority::High, Reviewed in Refinem...",,closed,5,2026-05-04,2026-04-21,2026-04-08,34,None,None,None
1,2017,Weekly Field Leader Talent Report [Product],"Ramu, NLG, Reviewed in Refinement",,closed,8,2026-05-04,2026-04-21,2026-04-08,34,None,None,None
2,2018,Weekly Printing FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",,closed,3,2026-05-04,2026-04-21,2026-04-08,34,None,None,None
3,2024,Field Leader Report Language Enhancements and ...,"Zagadou, NLG, ODM, Reviewed in Refinement, man...",,closed,1,2026-05-04,2026-04-21,2026-04-08,34,si_fl_top_ubs_ip_tracking,fl_recruitment,modified
4,2031,Weekly SBL FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",,closed,3,2026-05-04,2026-04-21,2026-04-08,34,si_fl_ubs_fa_red_flag_printing_wkly,fl_ubs_fa,new
5,2031,Weekly SBL FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",,closed,3,2026-05-04,2026-04-21,2026-04-08,34,si_fl_ubs_fa_red_flag_printing_wkly_top,fl_ubs_fa,new
6,2031,Weekly SBL FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",,closed,3,2026-05-04,2026-04-21,2026-04-08,34,si_fl_ubs_fa_red_flag_zero_sbl_wkly,fl_ubs_fa,new
7,2031,Weekly SBL FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",,closed,3,2026-05-04,2026-04-21,2026-04-08,34,si_fl_ubs_fa_red_flag_zero_sbl_wkly_top,fl_ubs_fa,new
8,2044,Bug Fix | Replace deep links in Medallia insights,"Zagadou, NLG, Reviewed in Refinement",,closed,1,2026-05-04,2026-04-21,2026-04-08,34,None,None,None
9,2045,Bug Fix | Replace deep link in structured prod...,"sanket, NLG, Reviewed in Refinement",,closed,1,2026-05-04,2026-04-21,2026-04-08,34,None,None,None



Label breakdown:


Reviewed in Refinement    47
manually altered          40
NLG                       38
New Insight               29
ODM                       29
Data Swat                 26
IKG                       22
approved                  17
sanket                    12
simple                    11
prev_iteration_closure    10
Ramu                       9
Top Feature                7
Priority::High             6
Dip                        5
Indu                       5
Sridhar KN                 4
complex-material           4
Chintan                    4
Amol                       4
Non-Swat                   4
Trishul                    3
Dennis                     3
Zagadou                    3
Ashutosh                   2
complex-immaterial         2
bug                        2
Helen                      1
Vinjosh                    1
Sravan                     1
Name: labels, dtype: int64


Rows labelled 'Top Feature':


,id_x,title,labels,rule_name,change_type
40,2131,Finalize NLG - Backup Withholding Insights,"prev_iteration_closure, sanket, Data Swat, NLG...",si_withholding_ssn_tin_applied,modified
41,2131,Finalize NLG - Backup Withholding Insights,"prev_iteration_closure, sanket, Data Swat, NLG...",si_withholding_no_w8_w9_documentation,modified
42,2131,Finalize NLG - Backup Withholding Insights,"prev_iteration_closure, sanket, Data Swat, NLG...",si_withholding_w8_expired_invalid,modified
43,2131,Finalize NLG - Backup Withholding Insights,"prev_iteration_closure, sanket, Data Swat, NLG...",si_withholding_first_b_notice,modified
44,2131,Finalize NLG - Backup Withholding Insights,"prev_iteration_closure, sanket, Data Swat, NLG...",si_withholding_irs_c_notice,modified
45,2131,Finalize NLG - Backup Withholding Insights,"prev_iteration_closure, sanket, Data Swat, NLG...",si_withholding_second_b_notice,modified
46,2131,Finalize NLG - Backup Withholding Insights,"prev_iteration_closure, sanket, Data Swat, NLG...",si_withholding_tin_missing_invalid,modified



Rows labelled 'New Insight':


,id_x,title,labels,rule_name,change_type
2,2018,Weekly Printing FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",None,None
4,2031,Weekly SBL FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",si_fl_ubs_fa_red_flag_printing_wkly,new
5,2031,Weekly SBL FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",si_fl_ubs_fa_red_flag_printing_wkly_top,new
6,2031,Weekly SBL FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",si_fl_ubs_fa_red_flag_zero_sbl_wkly,new
7,2031,Weekly SBL FL Talent Insight,"Ramu, Data Swat, IKG, NLG, New Insight, ODM, R...",si_fl_ubs_fa_red_flag_zero_sbl_wkly_top,new
11,2057,Upcoming Medicare Eligibility,"Indu, prev_iteration_closure, NLG, New Insight...",si_upcoming_medicare_eligibility,new
14,2081,Upcoming Social Security Eligibility,"sanket, NLG, New Insight, Non-Swat, ODM, Revie...",si_upcoming_social_security_eligibility_full_plus,new
15,2081,Upcoming Social Security Eligibility,"sanket, NLG, New Insight, Non-Swat, ODM, Revie...",si_upcoming_social_security_eligibility_full,new
16,2081,Upcoming Social Security Eligibility,"sanket, NLG, New Insight, Non-Swat, ODM, Revie...",si_upcoming_social_security_eligibility_first,new
17,2085,New Insight | Externally Held Annuities,"Dip, Data Swat, IKG, NLG, New Insight, ODM, Re...",si_yodlee_mtp_external_annuities,new


In [6]:
# ── Cell 6: Generate the executive summary ────────────────────────────────

print(f'Calling {MODEL_NAME} \u2026 this may take up to 30 seconds.')

SUMMARY_TEXT = generate_summary(df_current, next_month_df=df_next_month)

print('\n' + '=' * 80)
print(f'Executive Summary \u2013 {SELECTED_LABEL}')
print('=' * 80 + '\n')
print(SUMMARY_TEXT)

Calling gpt-4.1 … this may take up to 30 seconds.

Executive Summary – May 2026

1. Executive Summary:
- What materially changed this month: Multiple new client and field leader insights were launched, including eligibility alerts for Medicare and Social Security, externally held annuities, and asset movement tracking. Significant enhancements to reporting, validation frameworks, and bug fixes improved data quality and operational reliability.
- Why it matters to the business: These changes provide advisors and leaders with actionable intelligence, enabling proactive client engagement and retention. Improved reporting and platform stability support better decision-making and reduce operational risk.

2. Top Feature (or Insight) of the Month:
- Insight name: Finalize NLG - Backup Withholding Insights
- Brief description / example of narrative: Provides advisors with timely alerts when backup withholding is applied to client accounts, based on SSN/TIN status.
- Main benefit / value: Enab

In [7]:
# ── Cell 7: Save to .txt file (optional) ──────────────────────────────────

content  = build_save_content(SUMMARY_TEXT, SELECTED_LABEL)
filename = f'executive_summary_{SELECTED_MONTH}.txt'

with open(filename, 'w', encoding='utf-8') as f:
    f.write(content)

print(f'Saved \u2192 {filename}')

Saved → executive_summary_2026-05.txt
